# 04 - Paper parity: where every source figure stands

**What this notebook is for.** The replication contract: every figure and table from the two
source papers is either replicated, substituted with the difference documented, queued with
its cost, or declared out of scope with the reason. Canonical inventory:
`notes/plot_parity.md` (rendered below); this notebook also carries the results that live
nowhere else: the logit-lens split verdict and the preference (Elo) experiment.

**Key concepts.**
- *Battery*: our name for the scenario test set, the Anthropic paper's 12
  implicit-emotion prompts (its Table 2), plus our own held-out set of 12.
- *Logit lens*: projecting an internal direction through the model's output vocabulary matrix
  to see which tokens it up-weights.
- *Elo rating*: a single preference-strength score per activity, fitted (Bradley-Terry) from
  all pairwise A/B choices so that Elo gaps predict win probabilities.
- *Substitution*: same analysis, different tool (for example t-SNE for UMAP), always with the
  difference stated.

**Index.**
1. The inventory (both papers)
2. Logit lens (paper Table 1): instruct fails, base partially reproduces
3. Preferences (paper Figure 4, left half): probe activations vs revealed preference
4. Steering (paper Figure 4, right half): the causal test

## 1. The inventory

### Anthropic paper

| Item | What it shows | Status | Where / why |
|---|---|---|---|
| Figure 1 | Top-activating dataset snippets per emotion vector, external corpora | QUEUED (scaled down) | Needs a corpus sweep (LMSYS/Pile samples) with per-token projection; ~half a pod day; not gate-critical |
| Table 1 | Logit-lens top/bottom tokens per emotion vector | SPLIT: base partial, instruct negative | Section 2 below (rendered from `results/logit_lens_base_L33.json`, `logit_lens_base_L57.json`, `logit_lens_it_L33_normed.json`, `logit_lens_it_L57.json`; the section-2 cell prints the verdict): base vectors at layer 33 show affective token neighborhoods for about half the emotions; instruct vectors show none at layers 33 or 57. Final-norm scaling applied, softcapping ignored |
| Figure 2 | Probe x scenario cosine matrix, strong diagonal | DONE | notebooks/03, sections 1 and 3 (dual-model); our diagonals are weaker, which is a finding (TREE Q1.H2) |
| Table 2 | The 12 implicit-emotion scenarios | DONE | Used verbatim, src/emotion_vectors/probe_prompts.py |
| Figure 3 | Numerical-intensity template curves | DONE | notebooks/03, section 2 (dual-model): instruct tracks 11/11 registered directions, base 7/11 (computed there from `results/probe_sweep_it/activations.npz` and `results/probe_sweep/activations.npz`) |
| Figure 4 | Activity-preference Elo + steering shifts | REPRODUCES (after instrument fix) | Sections 3-4 below (TREE Q1.H3 [supported]). Probe-Elo holds under both probe instruments: max abs r 0.7013 with pre-fix probes and 0.6448 with post-fix probes, both at layer 33 and both above the 0.5 bar (paper 0.71-0.74; `results/preferences_it_chat_fixed/scores.json` and `scores_postfix_probes.json`, compared in section 3). Elo split 1727 vs -578 (same scores.json, printed under the section-3 figure). Steering valence-sign test 11/12 and 10/12, dose-responsive (`results/steering_it_fixed/scores.json`, `results/steering_it_a8_fixed/scores.json`; printed in section 4), confirmed at 10/12 for both doses with post-fix vectors (`results/steering_it_postfix/scores.json`, `results/steering_it_a8_postfix/scores.json`). The earlier negative verdicts were artifacts of a padding bug (Q1.H3.E4); plain format remains a dead instrument. Below-bar remainder: delta-vs-r coupling 0.32/0.23 at the two doses, 0.399/0.299 with the post-fix probe profile (paper 0.85; same steering scores files) |
| Figure 5 | Pairwise cosine similarity, clustered | DONE | notebooks/02, section 3 |
| Figure 6 | UMAP of k-means emotion clusters | SUB, DONE | notebooks/archive/09: t-SNE embedding instead of UMAP (dependency), identical k-means k=10; clusters interpretable (joy/hope family, calm/content family), matching the paper's qualitative result |
| Figure 7 | PC1/PC2 loading bars per emotion | DONE | notebooks/02, section 5: each model in its own valence-best/arousal-best component plane |
| Figure 8 | PC1/PC2 vs human valence/arousal ratings | SUB | We correlate against the NRC VAD lexicon (the replication's instrument), not Russell's 45-emotion ratings; documented in TREE Q1.H1.C1 |
| Figure 9 | Representational similarity across layers | DONE | notebooks/02, section 4 |
| Neutral-PC projection (methods) | Confound removal before probe use | DONE (late) | Missing from the reference code and our pipeline until 2026-07-21; E7 implements it |
| Appendix: token-level activation localization | Vectors activate on emotion-relevant story spans | QUEUED | Requires per-token projection; shared infrastructure with Q3 |
| Overview panels: reward-hacking steering | Steering shifts misalignment rates | OUT | Production alignment evals and steering infra; not reproducible here |

### Open replication (sinievanderben/emotion_experiment)

| Item | What it shows | Status | Where / why |
|---|---|---|---|
| fig1_cosine_similarity | Contrast-vector cosine heatmap | DONE | notebooks/02, section 3 |
| fig2_pca | PCA scatter + valence/arousal panels | DONE | notebooks/02, sections 1-2 |
| fig3_umap | UMAP colored by k-means cluster | SUB, DONE | Same t-SNE substitution as the paper's Figure 6; notebooks/archive/09 |
| fig_valence/arousal_trajectory | PC-correlation across layers, two models | DONE + extended | notebooks/02 (base); our base-vs-instruct comparison (results/emotion_geometry_correlations*.json) is the same plot family with a new finding (valence demotion, TREE Q1.H1.C2) |
| fig_cka (centered kernel alignment) | Cross-layer representation similarity | SUB | We use representational similarity analysis (correlation of pairwise-cosine structures) instead of CKA; same question, different similarity index; notebooks/02 section 4 |
| analyze_story_conditions | Same model, vectors from different story corpora | DONE | Our E5 comparison: 4B-corpus vs self-generated probes (notebooks/archive/07) |
| visualize_token_activations | Per-token projection along a sentence | QUEUED | Becomes Q3's core infrastructure (per-token trajectories) |

### Beyond the main results: the complete census

The Anthropic paper contains **86 numbered figures and 16 tables** in total; the table above
covers the main results only. The complete item-by-item census (audited 2026-07-22) lives in
`notes/plot_parity.md`. Summary: ~11 done or substituted, ~74 queued (appendix variants of one
per-token infrastructure, Gemma analogues of transcript illustrations, and base-vs-instruct
proxies for the post-training panels), ~15 out, all on a single blocker class (blackmail /
reward-hacking / sycophancy rollouts and on-policy Claude transcript corpora, e.g. Figures
26-35, 66-68, 76-79). Top queued items by value: the steering delta-log-prob test (Figures
52-53, the causal leg we have never run), the post-training proxy panels (Figures 36-39, 84,
Table 16; the paper's own quantities for our instruction-tuning finding C2), and self- vs
other-speaker structure (Figures 17-19, 59).

## Maintenance

Update this table whenever a QUEUED item lands or a new figure appears in
either source. The notebook restyle (plotly, skimmable cells) references this
inventory so each notebook states which paper figure it corresponds to.


## 2. Logit lens (paper Table 1): instruct fails, base partially reproduces

In [1]:
# this cell renders the logit-lens token tables for both models, with a layer toggle: instruct (fails) then base (partial)
import json
from pathlib import Path

import plotly.graph_objects as go

from emotion_vectors.artifacts import fetch  # local results/ first, HF otherwise

ROOT = Path("..")
STRONG_BASE = [
    "happy",
    "proud",
    "desperate",
    "angry",
    "guilty",
]  # affective neighborhoods, judged by eye

# Only two layers were ever computed per model (33 and 57, final-norm scaled); the
# dropdown covers every available results/logit_lens_*.json, no other layers exist.
LENS_FILES = {
    "gemma-4-31b-it": {33: "logit_lens_it_L33_normed.json", 57: "logit_lens_it_L57.json"},
    "gemma-4-31b (base)": {33: "logit_lens_base_L33.json", 57: "logit_lens_base_L57.json"},
}
LENS_DEFAULT_LAYER = {"gemma-4-31b-it": 57, "gemma-4-31b (base)": 33}


def lens_table(model_label: str) -> None:
    layer_files = LENS_FILES[model_label]
    layers_available = sorted(layer_files)
    default_layer = LENS_DEFAULT_LAYER[model_label]
    titles = {}
    fig = go.Figure()
    for layer in layers_available:
        lens = json.loads(fetch(layer_files[layer]).read_text())
        assert lens["layer"] == layer
        rows = [(e, ", ".join(t["up"]), ", ".join(t["down"])) for e, t in lens["table"].items()]
        fig.add_trace(
            go.Table(
                header=dict(
                    values=["emotion", "top up-weighted tokens", "top down-weighted tokens"],
                    align="left",
                ),
                cells=dict(values=list(zip(*rows)), align="left", height=26),
                visible=layer == default_layer,
            )
        )
        titles[layer] = (
            f"Logit lens, {model_label}, layer {layer} ({lens['note']})"
            "<br><sup>maps to: Anthropic Table 1; the dropdown covers both computed layers "
            f"(33 and 57, files results/{layer_files[33]} and results/{layer_files[57]}); "
            "no other layers were computed</sup>"
        )
    fig.update_layout(
        title=titles[default_layer],
        updatemenus=[
            dict(
                buttons=[
                    dict(
                        label=f"layer {layer}",
                        method="update",
                        args=[
                            {"visible": [lyr == layer for lyr in layers_available]},
                            {"title.text": titles[layer]},
                        ],
                    )
                    for layer in layers_available
                ],
                active=layers_available.index(default_layer),
                x=1.0,
                xanchor="right",
                y=1.16,
                yanchor="bottom",
            )
        ],
        height=470,
        margin=dict(t=90, b=10),
    )
    fig.show()


lens_table("gemma-4-31b-it")
lens_table("gemma-4-31b (base)")
print(
    "verdict, gemma-4-31b-it: no emotion-word neighborhoods at layer 33 or 57; Table 1 does not reproduce"
)
print(
    f"verdict, gemma-4-31b (base): affective neighborhoods for {len(STRONG_BASE)}/12 at layer 33 ({', '.join(STRONG_BASE)}); valence-consistent down-lists for most others"
)

verdict, gemma-4-31b-it: no emotion-word neighborhoods at layer 33 or 57; Table 1 does not reproduce
verdict, gemma-4-31b (base): affective neighborhoods for 5/12 at layer 33 (happy, proud, desperate, angry, guilty); valence-consistent down-lists for most others


<details><summary><b>How to read these tables</b></summary>

The paper's Table 1 shows each emotion vector up-weighting related words (sad toward grief, tears). Both tables apply the final-normalization scaling (Gemma's 1+w convention); the documented simplification is that logit softcapping is ignored. Each table's dropdown switches between the two computed layers (33 and 57); the lens was never run at any other layer, so those are the only views that exist, not a selection from a sweep.

- **Instruct (top table, layer 57 default)**: unrelated fragments everywhere, at both tested layers (33 in `results/logit_lens_it_L33_normed.json`, 57 in `results/logit_lens_it_L57.json`, both in the dropdown). A robust negative.
- **Base (bottom table, layer 33 default)**: clear affective neighborhoods for about half the emotions at layer 33 (happy toward delightful/wonderful, angry toward vicious/angrily, desperate toward misery/wretched, guilty toward conceal/incriminating), and valence-consistent down-lists for most others (sad down-weights charming/fabulous, calm down-weights brutal/vicious). Layer 57 (`results/logit_lens_base_L57.json`, in the dropdown) is similar but noisier. A partial positive, judged qualitatively, no registered quantitative bar.

The split matters: the same extraction pipeline yields vocabulary-aligned directions on the base model and junk on the instruct model. This is the third independent signature (with the probe-battery failure and the valence demotion, TREE Q1.H1.C2) that instruction tuning buries the affect representation under non-affective structure.

</details>

## 3. Preferences (paper Figure 4, left half): probe activations vs revealed preference

The paper's Figure 4 shows (left) per-emotion probe correlations with activity-preference
Elo ratings and (right) steering shifting those preferences. We reproduce the left half:
Elo from Bradley-Terry over all ordered activity pairs (TREE Q1.H3), on our self-authored
64-activity list (the paper's is unpublished; ours follows its 8 named categories). The
steering half is section 4. All data in sections 3-4 comes from the padding-FIXED collections (TREE Q1.H3.E4); the compromised originals are archived in the results history and on the dataset card. Both prompt formats shown: the paper's exact plain
format, and the model's own chat template (the E4 lesson: plain formatting is
out-of-distribution for an instruct model). The section closes with an instrument-sensitivity
check: the same chat-arm activations rescored against the post-fix probe set
(`results/preferences_it_chat_fixed/scores_postfix_probes.json`), so the headline correlation
is stated as a range over probe instruments, not a single number.

In [2]:
# this cell renders, per collected arm, the paper's Figure 4 left half (probe-Elo bars + best-probe
# scatter), with a slider over every scored probe layer; default is the registered best layer
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ARMS = [
    ("plain (paper's exact format)", fetch("preferences_it_fixed")),
    ("chat template", fetch("preferences_it_chat_fixed")),
]
CATEGORY_COLORS = {
    "helpful": "#2ca02c",
    "engaging": "#17becf",
    "social": "#1f77b4",
    "self_curiosity": "#9467bd",
    "neutral": "#7f7f7f",
    "aversive": "#8c564b",
    "misaligned": "#ff7f0e",
    "unsafe": "#d62728",
}

for arm_label, arm_dir in ARMS:
    scores_file = arm_dir / "scores.json"
    if not scores_file.exists():
        print(f"[{arm_label}] not collected/scored yet, skipped")
        continue
    s = json.loads(scores_file.read_text())
    elo_vals = np.array([a["elo"] for a in s["elo_per_activity"]])
    cats = [a["category"] for a in s["elo_per_activity"]]
    layers_scored = [b["layer"] for b in s["probe_elo_by_layer"]]
    default_idx = layers_scored.index(s["p2_best_layer"])
    n_traces_per_layer = 1 + len(CATEGORY_COLORS)  # bar + one scatter per category

    fig = make_subplots(
        rows=1,
        cols=2,
        column_widths=[0.55, 0.45],
        horizontal_spacing=0.09,
        subplot_titles=(
            "per-probe correlation with Elo (sorted)",
            "best probe at the selected layer vs Elo",
        ),
    )
    slider_steps = []
    for li, by_layer in enumerate(s["probe_elo_by_layer"]):
        r = np.array(by_layer["per_probe_r"])
        order = np.argsort(r)
        names = [s["probe_emotions_matched"][i] for i in order]
        act = np.array(by_layer["best_probe_activation_per_activity"])
        best_probe = s["probe_emotions_matched"][by_layer["argmax_probe_index"]]
        is_default = li == default_idx
        fig.add_bar(
            x=list(range(len(r))),
            y=r[order],
            marker_color="#4878a8",
            row=1,
            col=1,
            showlegend=False,
            visible=is_default,
        )
        for cat in CATEGORY_COLORS:
            idx = [i for i, c in enumerate(cats) if c == cat]
            fig.add_scatter(
                x=act[idx],
                y=elo_vals[idx],
                mode="markers",
                name=cat,
                marker=dict(color=CATEGORY_COLORS[cat], size=9),
                row=1,
                col=2,
                visible=is_default,
            )
        tickvals = list(range(0, len(r), 8))
        title = (
            f"Preference Elo vs emotion probes, gemma-4-31b-it, {arm_label}, "
            f"layer {by_layer['layer']}"
            + (" (registered best layer)" if is_default else "")
            + f": best probe '{best_probe}', max |r|={by_layer['max_abs_r']}"
            "<br><sup>maps to: Anthropic Figure 4, top-left panels "
            "(steering half is Q1.H3.E2, gated; activity list is our documented substitution)</sup>"
        )
        slider_steps.append(
            dict(
                label=f"L{by_layer['layer']}",
                method="update",
                args=[
                    {"visible": [lj == li for lj in range(len(layers_scored)) for _ in range(n_traces_per_layer)]},
                    {
                        "title.text": title,
                        "xaxis.tickvals": tickvals,
                        "xaxis.ticktext": [names[i] for i in tickvals],
                    },
                ],
            )
        )
        if is_default:
            fig.update_xaxes(
                tickvals=tickvals,
                ticktext=[names[i] for i in tickvals],
                tickangle=45,
                tickfont=dict(size=9),
                row=1,
                col=1,
            )
            fig.update_layout(title=title)
    fig.update_yaxes(title_text="Pearson r with Elo", row=1, col=1)
    fig.update_xaxes(title_text="probe activation (cosine)", row=1, col=2)
    fig.update_yaxes(title_text="Elo rating", row=1, col=2)
    fig.update_layout(
        sliders=[
            dict(
                active=default_idx,
                steps=slider_steps,
                currentvalue=dict(prefix="probe layer: "),
                y=-0.62,
                len=0.5,
                x=0.25,
            )
        ],
        height=540,
        legend=dict(orientation="h", y=-0.32),
        margin=dict(t=80, b=110),
    )
    fig.show()
    print(
        f"[{arm_label}] P1 {'PASS' if s['p1_pass'] else 'FAIL'} "
        f"(positive {s['p1_positive_mean']:.0f} vs negative {s['p1_negative_mean']:.0f}); "
        f"P2 {'PASS' if s['p2_pass'] else 'FAIL'} "
        f"(max |r|={s['p2_max_abs_r']} bar 0.5, organization r={s['p2_valence_organization_r']}, "
        f"perm p={s['p2_valence_organization_perm_p']})"
    )

[plain (paper's exact format)] P1 PASS (positive 1002 vs negative 1002); P2 FAIL (max |r|=0.4141 bar 0.5, organization r=-0.2293, perm p=0.0034)


[chat template] P1 PASS (positive 1727 vs negative -578); P2 PASS (max |r|=0.7013 bar 0.5, organization r=0.4189, perm p=0.0)


<details><summary><b>How to read these panels</b></summary>

Left panel (the paper's sorted bar chart): each bar is one of the 164 NRC-matched emotion probes;
its height is the Pearson correlation between that probe's activation on the activity tokens of
"How would you feel about {activity}?" and the activity's Elo. In the paper this is strongly
valence-organized (hostile at -0.74 to blissful at +0.71). Right panel (the paper's scatter):
the single best probe against Elo, one dot per activity, colored by the 8 activity categories.
In the paper unsafe/misaligned activities sit at the bottom (Elo ~583) and helpful/engaging at
the top (~2465).

The layer slider re-renders both panels at each scored probe layer (24, 30, 33, 36, from
`probe_elo_by_layer` in each arm's scores.json), so no single layer is privileged; the default
position is the registered best layer, and the title names the best probe at whichever layer
is selected.

Registered reads (declared in scripts/score_preferences.py before collection landed): P1, positive
categories out-Elo negative ones; P2, max |r| at least 0.5 with permutation-significant positive
valence organization. The verdict line under each figure reports both. Elo scale note: our anchor
is mean 1000 (the paper's anchor is unpublished), so only gaps and rankings are comparable.

</details>

In [3]:
# this cell states the instrument sensitivity of the probe-Elo result: max |r| per layer under
# the pre-fix and post-fix probe sets (chat arm; the padding fix is TREE Q1.H3.E4b)
chat_dir = fetch("preferences_it_chat_fixed")
prefix_scores = json.loads((chat_dir / "scores.json").read_text())
postfix_scores = json.loads((chat_dir / "scores_postfix_probes.json").read_text())

print("probe-Elo max |r| per layer, chat arm, by probe instrument (registered bar 0.5):")
print(f"{'layer':>6} {'pre-fix probes':>15} {'post-fix probes':>16}")
for pre_layer, post_layer in zip(
    prefix_scores["probe_elo_by_layer"], postfix_scores["probe_elo_by_layer"]
):
    assert pre_layer["layer"] == post_layer["layer"]
    print(f"{pre_layer['layer']:>6} {pre_layer['max_abs_r']:>15.4f} {post_layer['max_abs_r']:>16.4f}")
print(
    "sources: results/preferences_it_chat_fixed/scores.json (pre-fix probes) and "
    "scores_postfix_probes.json (post-fix probes)\n"
    "read: every layer clears the 0.5 bar under both instruments; the fix costs "
    f"|r| {prefix_scores['p2_max_abs_r']} to {postfix_scores['p2_max_abs_r']} at the best layer "
    f"({postfix_scores['p2_best_layer']}) but does not change any pass/fail verdict"
)

probe-Elo max |r| per layer, chat arm, by probe instrument (registered bar 0.5):
 layer  pre-fix probes  post-fix probes
    24          0.5948           0.5970
    30          0.6368           0.6223
    33          0.7013           0.6448
    36          0.6514           0.5658
sources: results/preferences_it_chat_fixed/scores.json (pre-fix probes) and scores_postfix_probes.json (post-fix probes)
read: every layer clears the 0.5 bar under both instruments; the fix costs |r| 0.7013 to 0.6448 at the best layer (33) but does not change any pass/fail verdict


## 4. Steering (paper Figure 4, right half): the causal test

If the vectors carry causal preference content, adding an emotion vector to the residual
stream during the A/B choice should shift the resulting Elo, and the shift should track the
correlational profile from section 3 (the paper reports r = 0.85 between the two, with mean
shifts of +212 for blissful and -303 for hostile steering). We steered all 12 battery
emotions at layer 33 during the full chat-format pair pass, at two doses spanning the
coherence-preserving range (TREE Q1.H3.E2; the dose escalation exists because the alpha
calibration statistic turned out to be fit-noise-dominated, documented in the tree).

In [4]:
# this cell renders the steering arm (paper Figure 4 bottom scatter): redistribution vs probe-Elo r, both doses
STEER_ARMS = [
    ("alpha=2", fetch("steering_it_fixed")),
    ("alpha=8", fetch("steering_it_a8_fixed")),
]

fig = go.Figure()
for arm_label, arm_dir in STEER_ARMS:
    s = json.loads((arm_dir / "scores.json").read_text())
    fig.add_scatter(
        x=[r["probe_elo_r"] for r in s["per_emotion"]],
        y=[r["mean_delta_elo_positive_categories"] for r in s["per_emotion"]],
        mode="markers+text",
        text=[r["emotion"] for r in s["per_emotion"]],
        textposition="top center",
        textfont=dict(size=9),
        name=f"{arm_label} (P2 r={s['p2_pearson_r']:+.2f})",
    )
fig.add_hline(y=0, line_color="gray", line_width=1)
fig.add_vline(x=0, line_color="gray", line_width=1)
fig.update_layout(
    title=(
        "Steering moves preferences in the valence-correct direction, gemma-4-31b-it, layer 33"
        "<br><sup>maps to: Anthropic Figure 4, bottom scatter (their r=0.85, mean deltas +212/-303; "
        "ours: valence-sign test 11/12 and 10/12, dose-responsive)</sup>"
    ),
    xaxis_title="probe-Elo correlation r (unsteered, from the chat arm)",
    yaxis_title="mean delta Elo, positive categories (steered - baseline)",
    height=480,
    margin=dict(t=80, b=10),
)
fig.show()
for arm_label, arm_dir in STEER_ARMS:
    s = json.loads((arm_dir / "scores.json").read_text())
    print(
        f"[{arm_label}] P1 {'PASS' if s['p1_pass'] else 'FAIL'}; "
        f"P2 {'PASS' if s['p2_pass'] else 'FAIL'} (r={s['p2_pearson_r']:+.3f}, p={s['p2_p_value']:.3f}); "
        f"P3 {'PASS' if s['p3_pass'] else 'FAIL'} ({s['p3_sign_agreements']}/12)"
    )

[alpha=2] P1 PASS; P2 FAIL (r=+0.322, p=0.307); P3 PASS (11/12)
[alpha=8] P1 PASS; P2 FAIL (r=+0.228, p=0.475); P3 PASS (10/12)


<details><summary><b>How to read this scatter</b></summary>

One dot per steered emotion, at both doses. The x axis is how well that emotion's probe
*predicted* preferences without steering (section 3's per-probe r). The y axis is how much
steering with that vector actually *moved* preferences, measured as the mean Elo change of
the positive-category activities (the identifiable redistribution statistic: with a
mean-anchored Elo, a uniform shift of all activities is invisible to pairwise choices by
construction, which is also why the paper's +212/-303 global means need their unpublished
anchoring convention to interpret).

What the data shows (padding-fixed collections, `results/steering_it_fixed/scores.json` and
`.../steering_it_a8_fixed/scores.json`): steering works in the valence-correct direction and
is dose-responsive. The valence-sign test passes at both doses (P3: 11/12 at alpha=2 and
10/12 at alpha=8, bar 9), redistribution grows with dose (loving +37.6 to +295.6 Elo,
desperate -13.0 to -78.9), and choice coherence is preserved (P1 passes at both doses).
What does *not* reproduce is the paper's fine-grained coupling: their scatter has slope
r = 0.85 between prediction and effect; ours stays below the 0.5 bar (P2: r = +0.322 and
+0.228, p > 0.3), partly reflecting the narrow x-range our 12 emotions span.

Registered reads (TREE Q1.H3.E2, amended for the anchor degeneracy before scoring): P1
coherence, P2 correlation at bar 0.5, P3 valence-sign agreement at 9/12. Verdict: the causal
leg of Q1.H3 is SUPPORTED (C3 survived its gate); the earlier "causal null" verdict was an
artifact of the padding bug (Q1.H3.E4) and is retracted. An E4b re-steer with post-fix
vector directions confirms the result (P3 10/12 at both doses,
`results/steering_it_postfix/scores.json` and `.../steering_it_a8_postfix/scores.json`).
Scope limits: single steering layer (33), all-position additive steering.

</details>